In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("..")
TRAIN_DIR = PROJECT_ROOT / "student_resource" / "dataset" / "train"

s1 = pd.read_csv(
    TRAIN_DIR / "train_source1.tsv",
    sep="\t"
)

s2 = pd.read_csv(
    TRAIN_DIR / "train_source2.tsv",
    sep="\t"
)

s3 = pd.read_csv(
    TRAIN_DIR / "train_source3.tsv",
    sep="\t"
)

gt = pd.read_csv(
    TRAIN_DIR / "train_ground_truth.tsv",
    sep="\t"
)

gt["matched_entity_ids"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .astype(str)
)

In [2]:
# Split Source 1 entities, NOT individual pairs

train_s1_ids, val_s1_ids = train_test_split(
    s1["entity_id"],
    test_size=0.20,
    random_state=42
)

train_s1_ids = set(train_s1_ids)
val_s1_ids = set(val_s1_ids)

print("Training S1 entities:", len(train_s1_ids))
print("Validation S1 entities:", len(val_s1_ids))

Training S1 entities: 1765456
Validation S1 entities: 441365


In [3]:
gt_train = gt[
    gt["source1_entity_id"].isin(train_s1_ids)
].copy()

gt_val = gt[
    gt["source1_entity_id"].isin(val_s1_ids)
].copy()

print("Train GT:", gt_train.shape)
print("Validation GT:", gt_val.shape)

assert len(gt_train) + len(gt_val) == len(gt)

assert set(gt_train["source1_entity_id"]).isdisjoint(
    set(gt_val["source1_entity_id"])
)

print("Validation split looks good.")

Train GT: (1765456, 2)
Validation GT: (441365, 2)
Validation split looks good.


In [4]:
# Fast lookup by entity_id

s2_lookup = s2.set_index("entity_id")
s3_lookup = s3.set_index("entity_id")

print("S2 lookup:", len(s2_lookup))
print("S3 lookup:", len(s3_lookup))

S2 lookup: 5034616
S3 lookup: 5285603


In [5]:
def inspect_matches(gt_df, n=10, random_state=42):

    sample = gt_df.sample(
        n=min(n, len(gt_df)),
        random_state=random_state
    )

    rows = []

    for _, gt_row in sample.iterrows():

        s1_id = gt_row["source1_entity_id"]

        # Get S1 record
        s1_matches = s1.loc[
            s1["entity_id"] == s1_id
        ]

        if s1_matches.empty:
            continue

        s1_row = s1_matches.iloc[0]

        # Handle singleton / empty ground truth
        matched_ids = [
            x.strip()
            for x in str(gt_row["matched_entity_ids"]).split(",")
            if x.strip()
        ]

        # No matches → record singleton information
        if not matched_ids:
            rows.append({
                "s1_id": s1_id,
                "matched_id": None,
                "source": None,
                "s1_name": s1_row["business_name"],
                "matched_name": None,
                "s1_address": s1_row["business_address"],
                "matched_address": None,
                "s1_country": s1_row["country"],
                "matched_country": None
            })
            continue

        # Process actual matches
        for matched_id in matched_ids:

            if matched_id.startswith("S2-"):
                if matched_id not in s2_lookup.index:
                    continue

                source_row = s2_lookup.loc[matched_id]
                source = "S2"

            elif matched_id.startswith("S3-"):
                if matched_id not in s3_lookup.index:
                    continue

                source_row = s3_lookup.loc[matched_id]
                source = "S3"

            else:
                continue

            rows.append({
                "s1_id": s1_id,
                "matched_id": matched_id,
                "source": source,

                "s1_name": s1_row["business_name"],
                "matched_name": source_row["business_name"],

                "s1_address": s1_row["business_address"],
                "matched_address": source_row["business_address"],

                "s1_country": s1_row["country"],
                "matched_country": source_row["country"]
            })

    return pd.DataFrame(rows)

gt_train_positive = gt_train[
    gt_train["matched_entity_ids"].str.strip() != ""
].copy()

print("Positive S1 entities:", len(gt_train_positive))

sample_matches = inspect_matches(
    gt_train_positive,
    n=10,
    random_state=42
)

display(sample_matches)

Positive S1 entities: 1666794


,s1_id,matched_id,source,s1_name,matched_name,s1_address,matched_address,s1_country,matched_country
0,S1-619976953,S2-969391270,S2,Aubrey's Law Group,Aubrey's Group Law,"211 Hudson Oaks Drive, Weatherford, TX","Hudson Oaks Drive, WEATHERFORD, TX",US,US
1,S1-619976953,S2-227880043,S2,Aubrey's Law Group,AUBREYSLAWGROUP.COM,"211 Hudson Oaks Drive, Weatherford, TX","211- Hudson Oaks Dr, WEATHERFORD, TX",US,US
2,S1-619976953,S3-111621219,S3,Aubrey's Law Group,Aubrey's Law-Group,"211 Hudson Oaks Drive, Weatherford, TX","211 Hudson Oaks Drive, Weatherford, Texas",US,US
3,S1-619976953,S3-510010635,S3,Aubrey's Law Group,Acdecl's Law Group,"211 Hudson Oaks Drive, Weatherford, TX","211 Hudson Oaks Dr, Weatherford, Texas",US,US
4,S1-459774923,S2-653700998,S2,Om Global Private Limited,ओम ग्लोबल प्राइवेट लिमिटेड,"D-630, First Floor, Saraswati Vihar, Pitampura...","D-630, FIRST FLOOR, SARASWATI VIHAR, PITAMPURA...",India,India
5,S1-459774923,S2-552141946,S2,Om Global Private Limited,ओम ग्लोबल प्राइवेट लिमिटेड,"D-630, First Floor, Saraswati Vihar, Pitampura...","D-630, FIRST FLOOR, SARASWATI VIHAR, PITAMPURA...",India,India
6,S1-459774923,S2-782397785,S2,Om Global Private Limited,ॐ ग्लोबल प्राइवेट लिमिटेड,"D-630, First Floor, Saraswati Vihar, Pitampura...","D-630, DELHI, NORTH WEST, दिल्ली",India,India
7,S1-459774923,S3-860034295,S3,Om Global Private Limited,Om Global Limited Service,"D-630, First Floor, Saraswati Vihar, Pitampura...","#836 D-630, Delhi, North West, दिल्ली",India,India
8,S1-459774923,S3-166999437,S3,Om Global Private Limited,omglobal.com,"D-630, First Floor, Saraswati Vihar, Pitampura...","H.no 836 D-630, Delhi, North West, DL",India,India
9,S1-742492047,S2-577403192,S2,Vision Lakshmi Systems Private Limited,विजन लक्ष्मी सिस्टम्स प्राइवेट लिमिटेड,"Epo-09-005, Emaar Emerald Plaza, Sector 65, Go...","EPO-A-09-005, SADAR BAZAR, हरियाणा",India,India


In [6]:
high_match_entities = gt_train[
    gt_train["matched_entity_ids"].str.count(",") >= 5
]

print("Entities with 6+ matches:",
      len(high_match_entities))

display(
    inspect_matches(
        high_match_entities,
        n=10,
        random_state=42
    )
)

Entities with 6+ matches: 201794


,s1_id,matched_id,source,s1_name,matched_name,s1_address,matched_address,s1_country,matched_country
0,S1-849154489,S2-280690055,S2,Smart Range LLC,SMART-RANGE,"415 Simpson Road, Princeton, TX","TX, PRINCETON, 415- SIMPSON RD",US,US
1,S1-849154489,S2-83682533,S2,Smart Range LLC,SMART RANGE,"415 Simpson Road, Princeton, TX","415- SIMPSON ROAD, PRINCETON, TX",US,US
2,S1-849154489,S3-642774395,S3,Smart Range LLC,smart range llc,"415 Simpson Road, Princeton, TX","415C Simpson Road, Princeton, Texas",US,US
3,S1-849154489,S3-717047468,S3,Smart Range LLC,Smart Range LLC,"415 Simpson Road, Princeton, TX","Texas, Princeton, 415C Simpson Rd",US,US
4,S1-849154489,S3-153912268,S3,Smart Range LLC,Smart LLC Center,"415 Simpson Road, Princeton, TX","415C Simpson Rd, Princeton, Texas",US,US
...,...,...,...,...,...,...,...,...,...
59,S1-40977123,S2-685378944,S2,Bateman Anchor Aktiengesellschaft L.L.C.,Bateman Anchor Aktiengesellschaft L.L.C.,"2886 Holden Beach Road, Unit 55, Supply, NC","2886 HOLDEN BEACH ROAD, SUPPLY, NC",US,US
60,S1-40977123,S2-436465181,S2,Bateman Anchor Aktiengesellschaft L.L.C.,Bateman Anchor Áktiengesellschaft L.L.C.,"2886 Holden Beach Road, Unit 55, Supply, NC","2886 HOLDEN BEACH ROAD, SUPPLY, NC",US,US
61,S1-40977123,S2-250836978,S2,Bateman Anchor Aktiengesellschaft L.L.C.,BATEMAN ANCHOR AKTIENGESELLSCHAFT,"2886 Holden Beach Road, Unit 55, Supply, NC","2886 HOLDEN BEACH RD, SUPPLY, NC",US,US
62,S1-40977123,S3-823488984,S3,Bateman Anchor Aktiengesellschaft L.L.C.,Bateman Anchor Áktiengesellschaft L.L.C.,"2886 Holden Beach Road, Unit 55, Supply, NC","2886 Holden Beach Rd, # 55, Supply, North Caro...",US,US


In [8]:
# Create one lookup table for S2 + S3
s23 = pd.concat(
    [
        s2.assign(source="S2"),
        s3.assign(source="S3")
    ],
    ignore_index=True
)

s23_lookup = s23.set_index("entity_id")

print("Combined S2 + S3:", s23.shape)

sample_gt = gt_train_positive.sample(
    n=min(10000, len(gt_train_positive)),
    random_state=42
).copy()

pairs = (
    sample_gt[
        ["source1_entity_id", "matched_entity_ids"]
    ]
    .assign(
        matched_entity_id=lambda x:
            x["matched_entity_ids"].str.split(",")
    )
    .explode("matched_entity_id")
)

pairs["matched_entity_id"] = (
    pairs["matched_entity_id"]
    .str.strip()
)

print("Positive pairs:", len(pairs))

display(pairs.head())

pairs = pairs.merge(
    s1[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country"
        ]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left"
)

pairs = pairs.rename(columns={
    "business_name": "s1_name",
    "business_address": "s1_address",
    "country": "s1_country"
})

pairs = pairs.drop(columns=["entity_id"])

pairs = pairs.merge(
    s23[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
            "source"
        ]
    ],
    left_on="matched_entity_id",
    right_on="entity_id",
    how="left",
    suffixes=("", "_matched")
)

pairs = pairs.rename(columns={
    "business_name": "matched_name",
    "business_address": "matched_address",
    "country": "matched_country"
})

pairs = pairs.drop(columns=["entity_id"])

positive_stats = pd.DataFrame({
    "name_exact":
        pairs["s1_name"] == pairs["matched_name"],

    "address_exact":
        pairs["s1_address"] == pairs["matched_address"],

    "country_exact":
        pairs["s1_country"] == pairs["matched_country"]
})

display(
    positive_stats.mean()
    .sort_values(ascending=False)
)

print(
    "Number of positive pairs analyzed:",
    len(positive_stats)
)

Combined S2 + S3: (10320219, 5)
Positive pairs: 36427


,source1_entity_id,matched_entity_ids,matched_entity_id
1045373,S1-619976953,"S2-969391270,S2-227880043,S3-111621219,S3-5100...",S2-969391270
1045373,S1-619976953,"S2-969391270,S2-227880043,S3-111621219,S3-5100...",S2-227880043
1045373,S1-619976953,"S2-969391270,S2-227880043,S3-111621219,S3-5100...",S3-111621219
1045373,S1-619976953,"S2-969391270,S2-227880043,S3-111621219,S3-5100...",S3-510010635
956487,S1-459774923,"S2-653700998,S2-552141946,S2-782397785,S3-8600...",S2-653700998


country_exact    1.000000
name_exact       0.046559
address_exact    0.020946
dtype: float64

Number of positive pairs analyzed: 36427


In [9]:
from collections import Counter

match_counter = Counter()

for value in gt_train["matched_entity_ids"]:

    if not value:
        continue

    for entity_id in value.split(","):
        entity_id = entity_id.strip()

        if entity_id:
            match_counter[entity_id] += 1

reuse_counts = pd.Series(match_counter)

print("Unique matched S2/S3 IDs:", len(reuse_counts))

print("\nDistribution:")
display(reuse_counts.value_counts().sort_index().head(20))

print(
    "\nIDs appearing in multiple S1 entities:",
    (reuse_counts > 1).sum()
)

print(
    "Positive pairs with country mismatch:",
    (~positive_stats["country_exact"]).sum()
)

print(
    "Positive pairs with country match:",
    positive_stats["country_exact"].sum()
)

Unique matched S2/S3 IDs: 6111064

Distribution:


1    6111064
Name: count, dtype: int64


IDs appearing in multiple S1 entities: 0
Positive pairs with country mismatch: 0
Positive pairs with country match: 36427


In [11]:
# True matches where BOTH raw name and raw address differ

hard_examples = pairs[
    (pairs["s1_name"] != pairs["matched_name"]) &
    (pairs["s1_address"] != pairs["matched_address"])
].copy()

print("Positive pairs with both name and address different:",
      len(hard_examples))

display(
    hard_examples[
        [
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address",
            "s1_country",
            "matched_country"
        ]
    ].sample(
        n=min(30, len(hard_examples)),
        random_state=42
    )
)

Positive pairs with both name and address different: 33968


,s1_name,matched_name,s1_address,matched_address,s1_country,matched_country
26460,Rithm Care,Nylabrix,"66 Deep Root Drive, Westminster, VT","Vermont, 66-70 Deep Root Dr, Westminster",US,US
3575,Prime Properties Private Limited,Prime Properties Private Ltd,"No.51, 6Th Cross, Maruthi Hbcs Layout Basavesh...","60, 6TH CROSS, MARUTHI HBCS LAYOUT BASAVESHWAR...",India,India
25944,Eye Colonial Medicine Inc,Eye Colonial Médicine Inc,"108 Gum Road, Greenville, NC","136 GUM RD, GREENVILLE, NC",US,US
1993,Foundation Four Europe Ltd,Ltd Foundation Four Europe,"Bhagat Singh Cl, Sonipat, Haryana, Sonipat, C/...","C/O: DATAR SINGH, BHAGAT SINGH CL, SONIPAT, Ha...",India,India
28919,Superior Outdoor Corp,Superior Outdoor Corporation,"Bihar, Road No. 20, Kanahauli Vishunudutt, P.O...","BABAN BIGHA, ROAD NO. 20, KANAHAULI VISHUNUDUT...",India,India
35855,Seven Estate,Seven [Éstate],"C-54, Tarang Apartments, 19, I.P. Extn., Patpa...","DL, C-54, East Delhi",India,India
19375,Coimbatore Seeds Private Limited,Coimbatore Sbeds Private Limited,"Sf No 393 / 3, Gpy(V) Nehru Nagar, 3Rd Street,...","SF NO #393 / 3, GPY(V) NEHRU NAGAR, 3RD STREET...",India,India
268,Deekay & Associates,Deekay & Associates LLP,"New Delhi, Office No. 01, Ground Floor, G-5 Ab...","OFFICE NO. 01, GROUND FLOOR, G-5 ABUL FAZAL EN...",India,India
13522,Krishna Infradevelopers Private Limited,Krishna Infradevelopers Private,"67/1217, B G Road Nadakkavu Post, Kozhikode, K...","67/1217, B G ROAD NADAKKAVU POST, KOZHIKODE, K...",India,India
24023,"Fisher, Gosnell and Solorzano Oaktree LLC","Fisher, Gosnell and LLC Oaktree Solorzano","7431 Gadsby Square, Fairfax County, VA","ALEXANRIA, 7431 GADLBY SQUARE, VA",US,US


In [12]:
name_variations = pairs[
    (pairs["s1_name"] != pairs["matched_name"]) &
    (pairs["s1_address"] == pairs["matched_address"])
]

print("Name differs, address identical:",
      len(name_variations))

display(
    name_variations[
        [
            "s1_name",
            "matched_name",
            "s1_address"
        ]
    ].sample(
        n=min(20, len(name_variations)),
        random_state=42
    )
)

Name differs, address identical: 763


,s1_name,matched_name,s1_address
17722,Aegis (India) Constructions Pvt Ltd,Wextavo,"Plot No -149/2489 Mangala Vihar, Baramunda, Bh..."
12659,Pinnacle Bold Inc,Pinnacle,"1019 Mountain View Drive, Spring Hill, TN"
36092,Shyam Software,Shyam,"Manimuriyil House, Kottamuri P.O Thrikodithana..."
9602,Koch Detailing Inc.,Koch,"107 Strong Court, Huntsville, AL"
16127,"Milena Francisco, DMD, P.C.","Deltakor Labs formerly: Milena Francisco, DMD,...","38730 Gold Ridge Road, Pomeroy, OH"
29232,Sundrop Exports Private Limited,Orbidrexzeph,418 Behind Hotel Awadhinternational Ismailanj ...
36214,"Elise D. Garber, L.C.S.W., DDS PC","Wextavo née Elise D. Garber, L.C.S.W., DDS PC","2739 Columbus Drive, Williams, AZ"
6875,Ecogreen Buildcon Pvt Ltd,Evoorbikor doing business as Ecogreen Buildcon...,"Plot No. N-3/216, Irc Village, Bhubaneswar, Kh..."
3129,Kolkata Metro Ltd,Vionovi,"C/O B Sain N A F C A P 58 C I, Kolkata, Calcut..."
10906,Bright Analytics Collective,Lyrakelo Co,"2135 37th Avenue, Portland, OR"


In [13]:
address_variations = pairs[
    (pairs["s1_name"] == pairs["matched_name"]) &
    (pairs["s1_address"] != pairs["matched_address"])
]

print("Name identical, address differs:",
      len(address_variations))

display(
    address_variations[
        [
            "s1_name",
            "matched_name",
            "s1_address",
            "matched_address"
        ]
    ].sample(
        n=min(20, len(address_variations)),
        random_state=42
    )
)

Name identical, address differs: 1696


,s1_name,matched_name,s1_address,matched_address
13981,Neon Partners,Neon Partners,"League City, TX, 311 Green Oaks Drive",NaN
2854,Womens Health Clinic Partners,Womens Health Clinic Partners,"1122 Vermont Avenue, Dallas, TX","1122 Vermont Ave, PMB 8011, Dallas, Texas"
3298,BTL Giant,BTL Giant,"NC, 1456 Galloway Road, Charlotte",NaN
8052,"Cristal Young Tankers, Inc","Cristal Young Tankers, Inc","NC, 2511 Old Salisbury Road, Winston-salem","2511 OLD SALISBURY RD, WINSTON-SALEM, NC"
29805,XNL Biotechnologies,XNL Biotechnologies,"4120 Kolb Street, Houston, TX",NaN
31035,Obsidian LLC,Obsidian LLC,"W63N167 Washington Avenue, City Of Cedarburg, WI","W63N167 Washington Ave, CEDARBURG, WI"
15065,Congregation Emanuel,Congregation Emanuel,"1445 St Paul Street, Bellingham, WA","1445 St Paul St, Bellignham, Washington"
36173,Environmental Association,Environmental Association,"9944 Charles White Road, NY, Allen","9944 Charles White Rd, Allen, New York"
26109,Youth Services,Youth Services,"Russell, 209 Shaw Street, IA","209 SSAW STREET, RUSSELL, IA"
17146,Onyx,Onyx,"100 1/2 APT 1 Main Street, Olney, IL","400 1/2 APT 1 Main St, Olney, Illinois"


In [14]:
def null_safe_equal(a, b):
    return (
        a.fillna("__MISSING__").astype(str)
        ==
        b.fillna("__MISSING__").astype(str)
    )

In [15]:
positive_stats = pd.DataFrame({
    "name_exact":
        null_safe_equal(
            pairs["s1_name"],
            pairs["matched_name"]
        ),

    "address_exact":
        null_safe_equal(
            pairs["s1_address"],
            pairs["matched_address"]
        ),

    "country_exact":
        null_safe_equal(
            pairs["s1_country"],
            pairs["matched_country"]
        )
})

display(
    positive_stats.mean()
    .sort_values(ascending=False)
)

country_exact    1.000000
name_exact       0.046559
address_exact    0.020946
dtype: float64

In [16]:
name_valid = (
    pairs["s1_name"].notna() &
    pairs["matched_name"].notna()
)

address_valid = (
    pairs["s1_address"].notna() &
    pairs["matched_address"].notna()
)

print(
    "Name exact among pairs where both names exist:",
    (
        pairs.loc[name_valid, "s1_name"]
        ==
        pairs.loc[name_valid, "matched_name"]
    ).mean()
)

print(
    "Address exact among pairs where both addresses exist:",
    (
        pairs.loc[address_valid, "s1_address"]
        ==
        pairs.loc[address_valid, "matched_address"]
    ).mean()
)

Name exact among pairs where both names exist: 0.04655887116699152
Address exact among pairs where both addresses exist: 0.02190766050304353
